In [0]:
CATALOG = "capstone_supply_chain"

table_comments = {
    "gold.fact_purchase_orders": "One row per purchase order (PO) line, enriched with vendor and plant details. Includes quantity, unit price, line total, currency, order and expected delivery dates, status and plant.",
    "gold.agg_vendor_scorecard": "One row per vendor with performance metrics: on-time delivery percent, quality grade, defect rate in PPM, audit score, risk rating and a weighted composite score from 0 to 100 (higher is better). Also shows how many POs each vendor has and which currencies it trades in.",
    "gold.agg_vendor_spend_by_currency": "Total purchase order spend per vendor and per currency. One row per vendor and currency. Amounts in different currencies must never be added together.",
    "gold.agg_inventory_health": "Inventory health summary grouped by warehouse, part category and ABC class. Shows part counts, stock value in USD, units, and how many items are CRITICAL, REORDER, OVERSTOCK or HEALTHY.",
    "gold.dim_vendors": "Vendor master data: country, supply category, quality grade, risk rating, payment terms, certifications and contact email. One row per vendor.",
    "gold.dim_parts": "Parts master data: name, category, ABC class, unit cost in USD, lead time and hazardous flag. One row per part number.",
    "gold.dim_plants": "Plant master data: location, plant type, daily capacity, employees, warehouse code and ERP system. One row per plant.",
    "gold_consolidated.supply_chain_kpi_snapshot": "Single-row snapshot of headline supply chain KPIs: purchase order count, active vendors, average lead time, average vendor score, high-risk vendors, inventory value in USD, critical and reorder stock items, on-time delivery percent, pending deliveries and open quality issues. There is no total spend KPI because the data has five currencies.",
}

for table, comment in table_comments.items():
    spark.sql(f"COMMENT ON TABLE {CATALOG}.{table} IS '{comment}'")
    print(f"✅ Table comment: {table}")

column_comments = [
    ("gold.fact_purchase_orders", "line_total", "Total cost of the PO line (order_qty times unit_price), in the currency of the PO. Never add line totals across different currencies."),
    ("gold.fact_purchase_orders", "currency", "Currency code of the PO: USD, EUR, JPY, SEK or CAD."),
    ("gold.fact_purchase_orders", "lead_time_days", "Days between order_date and expected_delivery."),
    ("gold.fact_purchase_orders", "status", "Current PO status: Delivered, Confirmed, In Production, Pending Approval, In Transit or Quality Hold."),
    ("gold.agg_vendor_scorecard", "composite_score", "Weighted vendor score from 0 to 100: on-time delivery 30 percent, audit score 30 percent, quality (from defect rate) 20 percent, risk 20 percent. Higher is better."),
    ("gold.agg_vendor_scorecard", "defect_rate_ppm", "Defective parts per million units supplied. Lower is better. Below 200 is good, 200 to 350 is concerning, above 350 needs corrective action."),
    ("gold.agg_vendor_scorecard", "on_time_delivery_pct", "Percentage of deliveries from the vendor that arrived on time. Higher is better."),
    ("gold.agg_vendor_scorecard", "risk_rating", "Supplier risk: Low (safe), Medium (monitor) or High (action needed)."),
    ("gold.agg_vendor_spend_by_currency", "total_spend", "Sum of PO line totals for this vendor in this currency only."),
    ("gold.agg_inventory_health", "health_score", "Percentage of parts in HEALTHY stock status in this group. 100 means every part is healthy."),
    ("gold.agg_inventory_health", "critical_items", "Number of parts with current stock at or below safety stock. Needs immediate reorder."),
]

for table, column, comment in column_comments:
    safe = comment.replace("'", "\\'")   # escape any apostrophe
    spark.sql(f"ALTER TABLE {CATALOG}.{table} ALTER COLUMN {column} COMMENT '{safe}'")
    print(f"  ✅ {table}.{column}")

print("\n✅ Comments added")

In [0]:
spark.sql(f"""
    SELECT table_schema, table_name, comment
    FROM {CATALOG}.information_schema.tables
    WHERE table_schema IN ('gold', 'gold_consolidated')
    ORDER BY table_schema, table_name
""").show(truncate=60)

spark.sql(f"""
    SELECT table_name, column_name, comment
    FROM {CATALOG}.information_schema.columns
    WHERE table_schema IN ('gold', 'gold_consolidated') AND comment IS NOT NULL
    ORDER BY table_name, column_name
""").show(truncate=60)